# CREATE ML-MODEL

## LOAD DATA

In [1]:
import os
import pandas as pd
import duckdb
from functions import choose_features_target
from sklearn.model_selection import TimeSeriesSplit

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_features.parquet')
            """)

df_features = con.execute("SELECT * FROM train_delay").fetchdf()

## PREPROCESSING

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# define features groups
feature_scheme = {
    "categorical_one_hot": [
        "station_current", "train_name", "train_type", 
        "station_start", "station_dest", "station_role"
    ],
    "numeric_scaled": [
        "temperature", "precipitation_log", "wind_speed", 
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", 
        "month_sin", "month_cos", "travel_time", "dwell_time_planned", 
        "stop_index", "time_since_start_planned", "share_ride_time",
        "hist_delay_avg", "hist_delay_q90", "hist_delay_count"
    ],
    "binary_passthrough": [
        "precipitation_any", "feast", "weekend", 
        "departure_rush_morning", "departure_rush_evening"
    ]
}


### PREPROCESSOR ###

preprocessor = ColumnTransformer(
    transformers=[

        # categories: one-hot 
        # ("cat", OneHotEncoder(handle_unknown = "ignore", sparse_output = False, min_frequency = 0.0001), feature_scheme["categorical_one_hot"]),
        # numeric variables: standard scaling
        # ("num", StandardScaler(), feature_scheme["numeric_scaled"]),
        # binary variables: do nothing
        ("pass", "passthrough", feature_scheme["binary_passthrough"])
    ],
    # drop rest
    remainder = "drop")

In [4]:
# separate features and target
X, y = choose_features_target(df_features)

In [5]:
X_cat = X[["station_current", "train_name", "train_type", 
        "station_start", "station_dest", "station_role"]]

X_num = X[["temperature", "precipitation_log", "wind_speed", 
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", 
        "month_sin", "month_cos", "travel_time", "dwell_time_planned", 
        "stop_index", "time_since_start_planned", "share_ride_time",
        "hist_delay_avg", "hist_delay_q90", "hist_delay_count"]]

## SELECT VARIABLES

In [5]:
from sklearn.feature_selection import SelectKBest, f_regression

X_transformed = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()

# 3. SelectKBest auf die transformierten Daten anwenden
selector = SelectKBest(score_func=f_regression, k="all")
selector.fit(X_transformed, y)

# 4. Ergebnisse anzeigen
feature_scores = pd.DataFrame({'Feature': feature_names, 'Score': selector.scores_})
print(feature_scores.sort_values(by='Score', ascending=False).head(20))

                                 Feature          Score
128        num__time_since_start_planned  156237.898646
127                      num__stop_index  147897.101909
129                 num__share_ride_time  126506.232884
120                        num__hour_cos   46154.376978
119                        num__hour_sin   39555.533432
115              cat__station_role_start   37389.769140
125                     num__travel_time   19303.090633
113                cat__station_role_end   13734.648244
134         pass__departure_rush_evening    7698.145926
116                     num__temperature    5812.238870
132                        pass__weekend    4374.236183
133         pass__departure_rush_morning    4329.843961
92        cat__station_dest_Dortmund Hbf    4237.822450
124                       num__month_cos    3537.479773
114                cat__station_role_mid    3180.456519
89   cat__station_dest_Berlin Ostbahnhof    3121.429526
66        cat__station_start_Dresden Hbf    2661

## MODEL

In [6]:
### CHOOSE MODEL ###

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models_list = {
    "linear": LinearRegression(),
    
    "rf": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=50,  
        n_jobs=-1,
        random_state=42,
    ),
    
    "gb": GradientBoostingRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
    ),
}

## PIPELINE


In [27]:
from sklearn.pipeline import make_pipeline

pipeline_test = make_pipeline(
    preprocessor,
    models_list["rf"],
)

In [31]:
from sklearn.inspection import permutation_importance

X_test = X[:180000]
y_test = y[:180000]

pipeline_test.fit(X_test, y_test)


result = permutation_importance(
    pipeline_test, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)


sorted_importances_idx = result.importances_mean.argsort()[::-1]
importances = pd.DataFrame(
    result.importances_mean[sorted_importances_idx], 
    index=X.columns[sorted_importances_idx], 
    columns=['Importance_Mean']
)

print(importances)

                          Importance_Mean
weekend                          0.009377
feast                            0.004347
departure_rush_morning           0.002154
precipitation_any                0.000924
departure_rush_evening           0.000787
precipitation_log                0.000000
hist_delay_avg                   0.000000
train_type                       0.000000
station_current                  0.000000
station_dest                     0.000000
temperature                      0.000000
wind_speed                       0.000000
station_start                    0.000000
stops_total                      0.000000
travel_time                      0.000000
weekday_sin                      0.000000
weekday_cos                      0.000000
hist_delay_count                 0.000000
month_sin                        0.000000
month_cos                        0.000000
hist_delay_q90                   0.000000
hour_sin                         0.000000
hour_cos                         0

## CROSS-VALIDATION

In [8]:
# define time series cross-validator
ts_cv = TimeSeriesSplit(
    n_splits=5,
    gap = 50, # should i add a gap?
    max_train_size = 10000,
)

# get overview about splitting
for i, (train_index, test_index) in enumerate(ts_cv.split(df_features["stops_total"])):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")


# check if splitting worked as expected
all_splits = list(ts_cv.split(df_features))
train_0, test_0 = all_splits[0]
df_features.iloc[test_0]

Fold 0:
  Train: index=[296996 296997 296998 ... 306993 306994 306995]
  Test:  index=[307046 307047 307048 ... 614087 614088 614089]
Fold 1:
  Train: index=[604040 604041 604042 ... 614037 614038 614039]
  Test:  index=[614090 614091 614092 ... 921131 921132 921133]
Fold 2:
  Train: index=[911084 911085 911086 ... 921081 921082 921083]
  Test:  index=[ 921134  921135  921136 ... 1228175 1228176 1228177]
Fold 3:
  Train: index=[1218128 1218129 1218130 ... 1228125 1228126 1228127]
  Test:  index=[1228178 1228179 1228180 ... 1535219 1535220 1535221]
Fold 4:
  Train: index=[1525172 1525173 1525174 ... 1535169 1535170 1535171]
  Test:  index=[1535222 1535223 1535224 ... 1842263 1842264 1842265]


,ride_id,train_name,train_type,station_current,station_dest,departure_planned,departure_real,arrival_planned,arrival_real,temperature,...,stop_index,time_since_start_planned,share_ride_time,precipitation_any,precipitation_log,hist_delay_avg,hist_delay_q90,hist_delay_count,departure_rush_morning,departure_rush_evening
307046,42994,ICE 108,ICE,Köln Hbf,Köln Hbf,NaT,NaT,2024-08-21 15:33:00,2024-08-21 15:32:00,17.05,...,4,153.0,1.000000,1,0.182322,NaN,NaN,NaN,0,0
307047,42995,ICE 108,ICE,Karlsruhe Hbf,Köln Hbf,2024-08-22 13:00:00,2024-08-22 13:01:00,NaT,NaT,18.16,...,1,0.0,0.000000,0,0.000000,0.000000,0.0,0.0,0,0
307048,42995,ICE 108,ICE,Mannheim Hbf,Köln Hbf,2024-08-22 13:35:00,2024-08-22 13:40:00,2024-08-22 13:24:00,2024-08-22 13:23:00,18.29,...,2,24.0,0.156863,0,0.000000,0.000000,0.0,0.0,0,0
307049,42995,ICE 108,ICE,Wiesbaden Hbf,Köln Hbf,2024-08-22 14:39:00,2024-08-22 14:42:00,2024-08-22 14:31:00,2024-08-22 14:34:00,18.48,...,3,91.0,0.594771,0,0.000000,0.000000,0.0,0.0,0,0
307050,42995,ICE 108,ICE,Köln Hbf,Köln Hbf,NaT,NaT,2024-08-22 15:33:00,2024-08-22 15:35:00,18.46,...,4,153.0,1.000000,0,0.000000,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614085,86126,ICE 209,ICE,Hamburg Dammtor,Köln Hbf,2025-06-15 20:38:00,2025-06-15 21:02:00,2025-06-15 20:37:00,2025-06-15 21:01:00,19.89,...,2,8.0,0.031008,1,1.856298,8.448416,25.0,3441.0,0,0
614086,86126,ICE 209,ICE,Hamburg Hbf,Köln Hbf,2025-06-15 20:46:00,2025-06-15 21:11:00,2025-06-15 20:42:00,2025-06-15 21:07:00,19.89,...,3,13.0,0.050388,1,1.856298,7.883349,22.9,4372.0,0,0
614087,86126,ICE 209,ICE,Hamburg-Harburg,Köln Hbf,2025-06-15 20:56:00,2025-06-15 21:23:00,2025-06-15 20:55:00,2025-06-15 21:21:00,19.60,...,4,26.0,0.100775,1,2.208274,11.227273,30.0,2948.0,0,0
614088,86126,ICE 209,ICE,Bremen Hbf,Köln Hbf,2025-06-15 21:44:00,2025-06-15 22:10:00,2025-06-15 21:41:00,2025-06-15 22:07:00,19.02,...,5,72.0,0.279070,1,1.824549,11.627583,32.0,1936.0,0,0


## TRAINING AND EVALUATION

In [9]:
### EVALUATE MODEL ###
from sklearn.model_selection import cross_validate

def evaluate(pipeline, X, y, cv):
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
        },
        n_jobs=-1,
    )

    mae = -scores["test_mae"]
    rmse = -scores["test_rmse"]

    return {
    "MAE_mean": float(mae.mean()),
    "MAE_std": float(mae.std()),
    "RMSE_mean": float(rmse.mean()),
    "RMSE_std": float(rmse.std()),
    }



In [28]:
evaluate(pipeline_test, X, y, cv=ts_cv)


{'MAE_mean': 10.722160108496524,
 'MAE_std': 1.5685530644882941,
 'RMSE_mean': 17.068237659767053,
 'RMSE_std': 0.9823304145436902}

## SAVE MODEL

In [22]:
import joblib

# Suppose you already fitted your pipeline
pipeline_first.fit(X, y["delay"])

# Export to a file
joblib.dump(pipeline_first, "trained_delay_model.pkl")
print("Pipeline saved as 'trained_delay_model.pkl'")


Pipeline saved as 'trained_delay_model.pkl'
